# OLAP · 04 · BQ4 · When in the year do events and claims arrive?
**Owner: Ar** · Reviewer: Ana · Decision supported: **when to schedule surge claims staff and hold reserves** (claims operations manager)

| Query | Question | OLAP operation |
|---|---|---|
| 4.1 | Events by month and hazard | **Pivot** (`PIVOT`: hazard values become columns) |
| 4.2 | Loss by month and hazard | **Pivot** on a different measure |
| 4.3 | Claims by season and line of business | **Unpivot** claim columns into rows, then **pivot** by season |
| 4.4 | Claims per event and loss per claim, by hazard | **Roll-up** to hazard, ratio measures |

Claim counts are recorded for only some events. Every claims query reports how many events it is based on.

In [ ]:
%run ./00_olap_setup

### 4.1 · Pivot: events by start month × hazard

In [ ]:
q41 = olap("bq4_pivot_events_month_hazard", """
SELECT * FROM (
  SELECT month_number, month_name, season, hazard_group FROM v_event_loss)
PIVOT (count(*) FOR hazard_group IN ('Flood' AS flood, 'Hail' AS hail, 'Storm' AS storm, 'Cyclone' AS cyclone,
                                     'Bushfire' AS bushfire, 'Earthquake' AS earthquake))
ORDER BY month_number
""", show=False)
q41 = q41.fillna(0)
q41["total"] = q41[["flood", "hail", "storm", "cyclone", "bushfire", "earthquake"]].sum(axis=1)
display(q41)

### 4.2 · Pivot: normalised loss ($m) by start month × hazard

In [ ]:
q42 = olap("bq4_pivot_loss_month_hazard", """
SELECT * FROM (
  SELECT month_number, month_name, hazard_group, loss_m FROM v_event_loss)
PIVOT (round(sum(loss_m), 0) FOR hazard_group IN ('Flood' AS flood, 'Hail' AS hail, 'Storm' AS storm, 'Cyclone' AS cyclone,
                                                 'Bushfire' AS bushfire, 'Earthquake' AS earthquake))
ORDER BY month_number
""")

### 4.3 · Unpivot then pivot: claims by season × line of business
The fact stores one column per claim line. `UNPIVOT` turns those columns into rows (one row per event per line), which can then be pivoted by season. Only events with a claims breakdown are included.

In [ ]:
q43 = olap("bq4_claims_season_line", """
WITH lines AS (
  SELECT season, line, claims FROM (
    SELECT season, domestic_building_claims AS building, domestic_content_claims AS contents,
           domestic_motor_claims AS motor, domestic_other_claims AS other_domestic, commercial_claims AS commercial
    FROM v_event_loss WHERE domestic_building_claims IS NOT NULL)
  UNPIVOT (claims FOR line IN (building, contents, motor, other_domestic, commercial)))
SELECT * FROM lines
PIVOT (sum(claims) FOR season IN ('Summer' AS summer, 'Autumn' AS autumn, 'Winter' AS winter, 'Spring' AS spring))
ORDER BY line
""")
n_breakdown = spark.sql("SELECT count(*) FROM v_event_loss WHERE domestic_building_claims IS NOT NULL").first()[0]
print(f"Based on the {n_breakdown} events that have a claims breakdown by line.")

### 4.4 · Claims workload and cost per claim, by hazard
`loss_per_claim` uses only events that have **both** a loss and a claim count, so the two sums cover the same events.

In [ ]:
q44 = olap("bq4_claims_by_hazard", """
SELECT hazard_group, count(*) AS events, count(total_claims) AS events_with_claims,
       sum(total_claims) AS claims,
       round(sum(total_claims) / count(total_claims), 0) AS claims_per_event,
       round(1e6 * sum(CASE WHEN total_claims > 0 THEN loss_m END)
                 / sum(CASE WHEN loss_m IS NOT NULL THEN total_claims END), 0) AS loss_per_claim_aud,
       round(sum(motor_share * total_claims) / sum(CASE WHEN motor_share IS NOT NULL THEN total_claims END) * 100, 1) AS pct_motor_claims,
       round(avg(duration_days), 1) AS avg_duration_days
FROM (SELECT *, domestic_motor_claims / nullif(total_claims, 0) AS motor_share FROM v_event_loss)
GROUP BY hazard_group ORDER BY claims DESC NULLS LAST
""")

### 4.5 · Season roll-up (one line per season)

In [ ]:
q45 = olap("bq4_season_summary", """
SELECT season, count(*) AS events, round(sum(loss_m), 0) AS loss_m,
       round(100 * count(*) / (SELECT count(*) FROM v_event_loss), 1) AS pct_events,
       round(100 * sum(loss_m) / (SELECT sum(loss_m) FROM v_event_loss), 1) AS pct_loss,
       sum(total_claims) AS claims_recorded
FROM v_event_loss GROUP BY season
ORDER BY CASE season WHEN 'Summer' THEN 1 WHEN 'Autumn' THEN 2 WHEN 'Winter' THEN 3 ELSE 4 END
""")